In [ ]:
// discount_factory_example_v1.cs

using System;


// =========================================================
// ENUM DISCOUNT TYPE
// =========================================================

public enum DiscountType
{
    OrderDiscount,
    ItemDiscount,
    ComboDiscount
}


// =========================================================
// DOMAIN MODEL
// =========================================================

public class PromotionContext
{
    public string PromotionName { get; set; }

    public decimal Value { get; set; }

    public DiscountType Type { get; set; }
}


// =========================================================
// HANDLER CONTRACT
// =========================================================

public interface IDiscountHandler
{
    void Process(PromotionContext context);
}


// =========================================================
// ORDER DISCOUNT HANDLER
// =========================================================

public class OrderDiscountHandler : IDiscountHandler
{
    public void Process(PromotionContext context)
    {
        Console.WriteLine("apply order discount");

        var finalValue = context.Value * 0.9m;

        Console.WriteLine($"promotion: {context.PromotionName}");
        Console.WriteLine($"final value: {finalValue}");
    }
}


// =========================================================
// ITEM DISCOUNT HANDLER
// =========================================================

public class ItemDiscountHandler : IDiscountHandler
{
    public void Process(PromotionContext context)
    {
        Console.WriteLine("apply item discount");

        var finalValue = context.Value - 5;

        Console.WriteLine($"promotion: {context.PromotionName}");
        Console.WriteLine($"final value: {finalValue}");
    }
}


// =========================================================
// COMBO DISCOUNT HANDLER
// =========================================================

public class ComboDiscountHandler : IDiscountHandler
{
    public void Process(PromotionContext context)
    {
        Console.WriteLine("apply combo discount");

        Console.WriteLine($"promotion: {context.PromotionName}");
    }
}


// =========================================================
// FACTORY METHOD
// =========================================================

public static class DiscountHandlerFactory
{
    public static IDiscountHandler Create(DiscountType type)
    {
        switch (type)
        {
            case DiscountType.OrderDiscount:
                return new OrderDiscountHandler();

            case DiscountType.ItemDiscount:
                return new ItemDiscountHandler();

            case DiscountType.ComboDiscount:
                return new ComboDiscountHandler();

            default:
                throw new Exception("unsupported discount type");
        }
    }
}


// =========================================================
// DEMO
// =========================================================


var promotion = new PromotionContext
{
    PromotionName = "summer promo",
    Value = 100,
    Type = DiscountType.OrderDiscount
};


// factory tạo handler
var handler = DiscountHandlerFactory.Create(promotion.Type);

// xử lý discount
handler.Process(promotion);
  

apply order discount
promotion: summer promo
final value: 90.0


# mô tả ví dụ 1

## cấu trúc
```
DiscountType
     ↓
DiscountHandlerFactory
     ↓
IDiscountHandler
     ↓
handler cụ thể
```

## flow thực tế
```
payload
   ↓
detect discount type
   ↓
DiscountHandlerFactory.Create()
   ↓
IDiscountHandler
   ↓
handler.Process()
```

## cấu trúc thư mục
```
discounts
│
├─ domain
│  ├─ discount_type.cs
│  └─ promotion_context.cs
│
├─ handlers
│  ├─ idiscount_handler.cs
│  ├─ order_discount_handler.cs
│  ├─ item_discount_handler.cs
│  └─ combo_discount_handler.cs
│
├─ factories
│  └─ discount_handler_factory.cs
│
└─ demo
   └─ program.cs
```

## lưu ý
- nếu lớn hơn 5 loại discount => không dùng switch factory method => dictionary resolver
- khi handler có dependency => dùng di resolver (giống strategy)

In [1]:
using System;
using System.Threading.Tasks;


// =========================================================
// ENUM
// =========================================================

public enum DiscountType
{
    OrderDiscount,
    ItemDiscount,
    ComboDiscount
}


// =========================================================
// DOMAIN
// =========================================================

public class PromotionContext
{
    public string PromotionName { get; set; }

    public decimal Value { get; set; }

    public DiscountType Type { get; set; }
}


// =========================================================
// HANDLER CONTRACT
// =========================================================

public interface IDiscountHandler
{
    Task Create(PromotionContext context);

    Task Update(PromotionContext context);

    Task Delete(PromotionContext context);
}


// =========================================================
// BASE HANDLER (TEMPLATE METHOD)
// =========================================================

public abstract class DiscountHandlerBase : IDiscountHandler
{
    public async Task Create(PromotionContext context)
    {
        Console.WriteLine("create discount start");

        var finalValue = Calculate(context);

        await Save(context, finalValue);

        await AfterCreate(context, finalValue);

        Console.WriteLine("create discount done\n");
    }

    public async Task Update(PromotionContext context)
    {
        Console.WriteLine("update discount start");

        var finalValue = Calculate(context);

        await Save(context, finalValue);

        await AfterUpdate(context, finalValue);

        Console.WriteLine("update discount done\n");
    }

    public async Task Delete(PromotionContext context)
    {
        Console.WriteLine("delete discount");

        await Task.CompletedTask;
    }

    protected abstract decimal Calculate(PromotionContext context);

    protected virtual Task Save(PromotionContext context, decimal finalValue)
    {
        Console.WriteLine($"promotion: {context.PromotionName}");
        Console.WriteLine($"final value: {finalValue}");

        return Task.CompletedTask;
    }

    protected virtual Task AfterCreate(PromotionContext context, decimal finalValue)
    {
        return Task.CompletedTask;
    }

    protected virtual Task AfterUpdate(PromotionContext context, decimal finalValue)
    {
        return Task.CompletedTask;
    }
}


// =========================================================
// ORDER DISCOUNT
// =========================================================

public class OrderDiscountHandler : DiscountHandlerBase
{
    protected override decimal Calculate(PromotionContext context)
    {
        Console.WriteLine("apply order discount 10%");
        return context.Value * 0.9m;
    }
}


// =========================================================
// ITEM DISCOUNT
// =========================================================

public class ItemDiscountHandler : DiscountHandlerBase
{
    protected override decimal Calculate(PromotionContext context)
    {
        Console.WriteLine("apply item discount -5");
        return context.Value - 5;
    }
}


// =========================================================
// COMBO DISCOUNT
// =========================================================

public class ComboDiscountHandler : DiscountHandlerBase
{
    protected override decimal Calculate(PromotionContext context)
    {
        Console.WriteLine("apply combo discount");
        return context.Value;
    }

    protected override Task AfterCreate(PromotionContext context, decimal finalValue)
    {
        Console.WriteLine("combo extra logic after create");
        return Task.CompletedTask;
    }
}


// =========================================================
// FACTORY
// =========================================================

public static class DiscountHandlerFactory
{
    public static IDiscountHandler Create(DiscountType type)
    {
        return type switch
        {
            DiscountType.OrderDiscount => new OrderDiscountHandler(),
            DiscountType.ItemDiscount => new ItemDiscountHandler(),
            DiscountType.ComboDiscount => new ComboDiscountHandler(),
            _ => throw new Exception("unsupported discount type")
        };
    }
}


// =========================================================
// DEMO
// =========================================================


var promotion = new PromotionContext
{
    PromotionName = "summer promo",
    Value = 100,
    Type = DiscountType.OrderDiscount
};

var handler = DiscountHandlerFactory.Create(promotion.Type);

await handler.Create(promotion);

await handler.Update(promotion);

await handler.Delete(promotion);

create discount start
apply order discount 10%
promotion: summer promo
final value: 90.0
create discount done

update discount start
apply order discount 10%
promotion: summer promo
final value: 90.0
update discount done

delete discount


# mô tả ví dụ 2 

## cấu trúc
```
handlers
├─ abstractions
│  ├─ idiscount_handler.cs
│  └─ discount_handler_base.cs
│
├─ order_discount_handler.cs
├─ item_discount_handler.cs
└─ combo_discount_handler.cs
```